In [4]:
import os

from dotenv import load_dotenv

# Document Loader
from langchain_community.document_loaders import WebBaseLoader

# Text Splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embeddings
from langchain_huggingface import HuggingFaceEmbeddings

# Vector Store
from langchain_community.vectorstores import FAISS

# LLM
from langchain_groq import ChatGroq

In [5]:
load_dotenv()

os.environ["USER_AGENT"] = "News-RAG-Chatbot"

groq_api = os.getenv("GROQ_API_KEY")

print("Groq API Loaded Successfully ✅")

Groq API Loaded Successfully ✅


# 📰 Step 3: Load BBC News Website

In this step, we load the BBC News homepage using LangChain's WebBaseLoader and inspect the extracted content before preprocessing.

In [6]:
url = "https://www.bbc.com/news"

loader = WebBaseLoader(url)

documents = loader.load()

print(f"Total Documents : {len(documents)}")

Total Documents : 1


In [7]:
print(documents[0].metadata)

{'source': 'https://www.bbc.com/news', 'title': 'BBC News - Breaking news, video and the latest top stories from the U.S. and around the world', 'description': 'Visit BBC News for the latest news, breaking news, video, audio and analysis. BBC News provides trusted World, U.S. and U.K. news as well as local and regional perspectives. Also entertainment, climate, business, science, technology and health news.', 'language': 'en-GB'}


In [8]:
print(documents[0].page_content[:1500])

BBC News - Breaking news, video and the latest top stories from the U.S. and around the worldSkip to contentBritish Broadcasting CorporationHomeNewsFootball 2026SportBusinessTechnologyHealthCultureArtsTravelEarthAudioVideoLiveUS & CanadaUKAfricaAsiaAustraliaEuropeLatin AmericaMiddle EastIn PicturesBBC InDepthBBC VerifyHomeNewsUS & CanadaUKUK PoliticsEnglandN. IrelandN. Ireland PoliticsScotlandScotland PoliticsWalesWales PoliticsAfricaAsiaChinaIndiaAustraliaEuropeLatin AmericaMiddle EastIn PicturesBBC InDepthBBC VerifyFootball 2026SportBusinessWorld of BusinessTechnology of BusinessNYSE Opening BellTechnologyArtificial IntelligenceIntelligence RevolutionAI v the MindTech NowHealthCultureFilm & TVMusicArt & DesignStyleBooksEntertainment NewsArtsArts in MotionTravelDestinationsAfricaAntarcticaAsiaAustralia and PacificCaribbean & BermudaCentral AmericaEuropeMiddle EastNorth AmericaSouth AmericaWorld’s TableCulture & ExperiencesAdventuresThe SpeciaListEarthScienceNatural WondersClimate Solu

In [9]:
documents[0].metadata

{'source': 'https://www.bbc.com/news',
 'title': 'BBC News - Breaking news, video and the latest top stories from the U.S. and around the world',
 'description': 'Visit BBC News for the latest news, breaking news, video, audio and analysis. BBC News provides trusted World, U.S. and U.K. news as well as local and regional perspectives. Also entertainment, climate, business, science, technology and health news.',
 'language': 'en-GB'}

In [10]:
print(documents[0].page_content[:2000])

BBC News - Breaking news, video and the latest top stories from the U.S. and around the worldSkip to contentBritish Broadcasting CorporationHomeNewsFootball 2026SportBusinessTechnologyHealthCultureArtsTravelEarthAudioVideoLiveUS & CanadaUKAfricaAsiaAustraliaEuropeLatin AmericaMiddle EastIn PicturesBBC InDepthBBC VerifyHomeNewsUS & CanadaUKUK PoliticsEnglandN. IrelandN. Ireland PoliticsScotlandScotland PoliticsWalesWales PoliticsAfricaAsiaChinaIndiaAustraliaEuropeLatin AmericaMiddle EastIn PicturesBBC InDepthBBC VerifyFootball 2026SportBusinessWorld of BusinessTechnology of BusinessNYSE Opening BellTechnologyArtificial IntelligenceIntelligence RevolutionAI v the MindTech NowHealthCultureFilm & TVMusicArt & DesignStyleBooksEntertainment NewsArtsArts in MotionTravelDestinationsAfricaAntarcticaAsiaAustralia and PacificCaribbean & BermudaCentral AmericaEuropeMiddle EastNorth AmericaSouth AmericaWorld’s TableCulture & ExperiencesAdventuresThe SpeciaListEarthScienceNatural WondersClimate Solu

In [11]:
content = documents[0].page_content

print("Characters :", len(content))
print("Words :", len(content.split()))

Characters : 10814
Words : 1493


In [12]:
type(documents[0])

langchain_core.documents.base.Document

In [13]:
dir(documents[0])

['__abstractmethods__',
 '__annotations__',
 '__class__',
 '__class_getitem__',
 '__class_vars__',
 '__copy__',
 '__deepcopy__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__fields__',
 '__fields_set__',
 '__format__',
 '__ge__',
 '__get_pydantic_core_schema__',
 '__get_pydantic_json_schema__',
 '__getattr__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__iter__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__pretty__',
 '__private_attributes__',
 '__pydantic_complete__',
 '__pydantic_computed_fields__',
 '__pydantic_core_schema__',
 '__pydantic_custom_init__',
 '__pydantic_decorators__',
 '__pydantic_extra__',
 '__pydantic_extra_info__',
 '__pydantic_fields__',
 '__pydantic_fields_set__',
 '__pydantic_generic_metadata__',
 '__pydantic_init_subclass__',
 '__pydantic_on_complete__',
 '__pydantic_parent_namespace__',
 '__pydantic_post_init__',
 '__pydantic_private__',
 '__pydantic_root_model__

# ✂️ Step 4: Document Chunking

Large Language Models have context window limitations. Instead of embedding the entire webpage as a single document, we split it into smaller overlapping chunks.

Benefits:
- Better semantic search
- Higher retrieval accuracy
- Reduced token usage
- Improved answer quality

In [14]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", " ", ""]
)

In [15]:
chunks = splitter.split_documents(documents)

print(f"Total Chunks Created : {len(chunks)}")

Total Chunks Created : 14


In [16]:
print(chunks[0].page_content)

BBC News - Breaking news, video and the latest top stories from the U.S. and around the worldSkip to contentBritish Broadcasting CorporationHomeNewsFootball 2026SportBusinessTechnologyHealthCultureArtsTravelEarthAudioVideoLiveUS & CanadaUKAfricaAsiaAustraliaEuropeLatin AmericaMiddle EastIn PicturesBBC InDepthBBC VerifyHomeNewsUS & CanadaUKUK PoliticsEnglandN. IrelandN. Ireland PoliticsScotlandScotland PoliticsWalesWales PoliticsAfricaAsiaChinaIndiaAustraliaEuropeLatin AmericaMiddle EastIn PicturesBBC InDepthBBC VerifyFootball 2026SportBusinessWorld of BusinessTechnology of BusinessNYSE Opening BellTechnologyArtificial IntelligenceIntelligence RevolutionAI v the MindTech NowHealthCultureFilm & TVMusicArt & DesignStyleBooksEntertainment NewsArtsArts in MotionTravelDestinationsAfricaAntarcticaAsiaAustralia and PacificCaribbean & BermudaCentral AmericaEuropeMiddle EastNorth AmericaSouth AmericaWorld’s TableCulture & ExperiencesAdventuresThe SpeciaListEarthScienceNatural WondersClimate


In [17]:
chunks[0].metadata

{'source': 'https://www.bbc.com/news',
 'title': 'BBC News - Breaking news, video and the latest top stories from the U.S. and around the world',
 'description': 'Visit BBC News for the latest news, breaking news, video, audio and analysis. BBC News provides trusted World, U.S. and U.K. news as well as local and regional perspectives. Also entertainment, climate, business, science, technology and health news.',
 'language': 'en-GB'}

In [18]:
for i in range(3):
    print(f"\nChunk {i+1}")
    print("-" * 50)
    print(f"Characters : {len(chunks[i].page_content)}")
    print(chunks[i].page_content[:300])


Chunk 1
--------------------------------------------------
Characters : 995
BBC News - Breaking news, video and the latest top stories from the U.S. and around the worldSkip to contentBritish Broadcasting CorporationHomeNewsFootball 2026SportBusinessTechnologyHealthCultureArtsTravelEarthAudioVideoLiveUS & CanadaUKAfricaAsiaAustraliaEuropeLatin AmericaMiddle EastIn PicturesB

Chunk 2
--------------------------------------------------
Characters : 999
and PacificCaribbean & BermudaCentral AmericaEuropeMiddle EastNorth AmericaSouth AmericaWorld’s TableCulture & ExperiencesAdventuresThe SpeciaListEarthScienceNatural WondersClimate SolutionsSustainable BusinessGreen LivingAudioPodcast CategoriesRadioAudio FAQsVideoBBC MaestroDiscover the WorldLiveLi

Chunk 3
--------------------------------------------------
Characters : 998
Iran and Iraq over the coming days.19 hrs agoMiddle EastAustralia probes mystery space balls that washed up on beachOfficials are searching for the origins of six piec

In [19]:
type(chunks[0])

langchain_core.documents.base.Document

In [20]:
print(f"Original Documents : {len(documents)}")
print(f"Chunks Created     : {len(chunks)}")

Original Documents : 1
Chunks Created     : 14


# 🧠 Step 5: Generate Text Embeddings

Embeddings convert text into dense numerical vectors that capture semantic meaning.

Instead of matching exact keywords, semantic search compares vector similarity, allowing the system to retrieve contextually relevant information.

In [21]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully ✅")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully ✅


In [22]:
sample_text = chunks[0].page_content

vector = embedding_model.embed_query(sample_text)

print(type(vector))

<class 'list'>


In [23]:
print("Embedding Dimension:", len(vector))

Embedding Dimension: 384


In [24]:
vector[:15]

[0.05180592089891434,
 -0.09453608095645905,
 0.010052032768726349,
 -0.06614916771650314,
 0.07227117568254471,
 0.07911179214715958,
 -0.003302032360807061,
 -0.045856770128011703,
 -0.06018826365470886,
 0.043568648397922516,
 -0.0483039990067482,
 -0.029440078884363174,
 0.0027884854935109615,
 -0.007100337650626898,
 -0.02628009393811226]

In [25]:
sentence1 = "Artificial Intelligence is transforming healthcare."

sentence2 = "AI is changing the medical industry."

sentence3 = "The football match was exciting."

embedding1 = embedding_model.embed_query(sentence1)
embedding2 = embedding_model.embed_query(sentence2)
embedding3 = embedding_model.embed_query(sentence3)

In [26]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_1_2 = cosine_similarity([embedding1], [embedding2])[0][0]
similarity_1_3 = cosine_similarity([embedding1], [embedding3])[0][0]

print(f"Similarity (1 vs 2): {similarity_1_2:.3f}")
print(f"Similarity (1 vs 3): {similarity_1_3:.3f}")

Similarity (1 vs 2): 0.800
Similarity (1 vs 3): 0.048


In [27]:
print("Chunks Ready for Vector Database:", len(chunks))

Chunks Ready for Vector Database: 14


# 🗂️ Step 6: Create FAISS Vector Database

FAISS (Facebook AI Similarity Search) is a high-performance library used to store and search dense vectors efficiently.

Instead of searching raw text, FAISS searches vector representations, enabling fast semantic retrieval.

In [28]:
vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)

print("✅ FAISS Vector Store Created Successfully")

✅ FAISS Vector Store Created Successfully


In [29]:
type(vector_store)

langchain_community.vectorstores.faiss.FAISS

In [30]:
print("Total Indexed Chunks:", vector_store.index.ntotal)

Total Indexed Chunks: 14


In [31]:
vector_store.save_local("../faiss_index")

In [32]:
loaded_vector_store = FAISS.load_local(
    "../faiss_index",
    embeddings=embedding_model,
    allow_dangerous_deserialization=True
)

print("✅ FAISS Index Loaded Successfully")

✅ FAISS Index Loaded Successfully


In [33]:
print("Indexed Chunks:", loaded_vector_store.index.ntotal)

Indexed Chunks: 14


In [34]:
query = "What is today's top news?"

results = loaded_vector_store.similarity_search(
    query=query,
    k=3
)

print(f"Retrieved {len(results)} documents")

Retrieved 3 documents


In [35]:
for i, doc in enumerate(results, start=1):
    print(f"\n{'='*60}")
    print(f"Result {i}")
    print(f"{'='*60}")
    print(doc.page_content[:500])


Result 1
as Morocco become the first team to reach the World Cup quarter-finals, with co-hosts Canada exiting the competition.Most watched1BBC confronts head of Meta in India over child sex abuse ads2Why dangerous teen sex choking is raising alarm3Funeral of Iran's former supreme leader 'intensely political moment'4Watch: Taylor Swift and Travis Kelce wed in NYC - here's how the day unfolded5BBC in Tehran as mourners gather for former supreme leader's funeralAlso in newsNigeria says two nationals killed 

Result 2
"all the girls with ruffled socks and chubby cheeks".20 hrs agoTennisHow Iran's new regime is very different to what came beforeKhamenei's funeral is another reminder of the change Iran has seen, but what does its new leadership want?11 hrs agoBBC InDepthEvacuations in Guam as super typhoon Bavi approachesThe storm is forecast to bring winds in excess of 160mph and waves nearly 11m high when it makes landfall on Monday.7 hrs agoScience & EnvironmentLIVEWorld Cup 2026: Englan

In [36]:
for i, doc in enumerate(results, start=1):
    print(f"\nResult {i} Metadata")
    print(doc.metadata)


Result 1 Metadata
{'source': 'https://www.bbc.com/news', 'title': 'BBC News - Breaking news, video and the latest top stories from the U.S. and around the world', 'description': 'Visit BBC News for the latest news, breaking news, video, audio and analysis. BBC News provides trusted World, U.S. and U.K. news as well as local and regional perspectives. Also entertainment, climate, business, science, technology and health news.', 'language': 'en-GB'}

Result 2 Metadata
{'source': 'https://www.bbc.com/news', 'title': 'BBC News - Breaking news, video and the latest top stories from the U.S. and around the world', 'description': 'Visit BBC News for the latest news, breaking news, video, audio and analysis. BBC News provides trusted World, U.S. and U.K. news as well as local and regional perspectives. Also entertainment, climate, business, science, technology and health news.', 'language': 'en-GB'}

Result 3 Metadata
{'source': 'https://www.bbc.com/news', 'title': 'BBC News - Breaking news, 

# 🔍 Step 7: Create the Retriever

A retriever is responsible for fetching the most relevant document chunks from the vector database.

Instead of manually performing similarity search, we use LangChain's retriever interface, which can later be plugged directly into a RAG chain.

In [37]:
retriever = loaded_vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

print("Retriever created successfully ✅")

Retriever created successfully ✅


In [38]:
type(retriever)

langchain_core.vectorstores.base.VectorStoreRetriever

In [39]:
query = "What are the latest news headlines?"

retrieved_docs = retriever.invoke(query)

print(f"Retrieved {len(retrieved_docs)} documents")

Retrieved 3 documents


In [40]:
for i, doc in enumerate(retrieved_docs, start=1):
    print("=" * 70)
    print(f"Result {i}")
    print("=" * 70)
    print(doc.page_content[:500])
    print()

Result 1
News ÌgbòBBC News Japanese 日本語BBC News GahuzaBBC News PidginBBC News SomaliBBC News SwahiliBBC News தமிழ் (Tamil)BBC News తెలుగు (Telugu)BBC News ትግርኛ (Tigrinya)BBC News اردو (Urdu)BBC News YorùbáBBC News World ServiceFollow BBC on:Terms of UseSubscription TermsAbout the BBCPrivacy PolicyCookiesAccessibility HelpContact the BBCAdvertise with usDo not share or sell my infoBBC.com Help & FAQsContent IndexSet Preferred SourceCopyright 2026 BBC. All rights reserved. The BBC is not responsible for th

Result 2
as Morocco become the first team to reach the World Cup quarter-finals, with co-hosts Canada exiting the competition.Most watched1BBC confronts head of Meta in India over child sex abuse ads2Why dangerous teen sex choking is raising alarm3Funeral of Iran's former supreme leader 'intensely political moment'4Watch: Taylor Swift and Travis Kelce wed in NYC - here's how the day unfolded5BBC in Tehran as mourners gather for former supreme leader's funeralAlso in newsNigeria says t

In [41]:
for i, doc in enumerate(retrieved_docs, start=1):
    print(f"\nDocument {i} Metadata")
    print(doc.metadata)


Document 1 Metadata
{'source': 'https://www.bbc.com/news', 'title': 'BBC News - Breaking news, video and the latest top stories from the U.S. and around the world', 'description': 'Visit BBC News for the latest news, breaking news, video, audio and analysis. BBC News provides trusted World, U.S. and U.K. news as well as local and regional perspectives. Also entertainment, climate, business, science, technology and health news.', 'language': 'en-GB'}

Document 2 Metadata
{'source': 'https://www.bbc.com/news', 'title': 'BBC News - Breaking news, video and the latest top stories from the U.S. and around the world', 'description': 'Visit BBC News for the latest news, breaking news, video, audio and analysis. BBC News provides trusted World, U.S. and U.K. news as well as local and regional perspectives. Also entertainment, climate, business, science, technology and health news.', 'language': 'en-GB'}

Document 3 Metadata
{'source': 'https://www.bbc.com/news', 'title': 'BBC News - Breaking 

In [42]:
retriever_5 = loaded_vector_store.as_retriever(
    search_kwargs={"k": 5}
)

docs = retriever_5.invoke("Technology news")

print("Retrieved Documents:", len(docs))

Retrieved Documents: 5


In [43]:
manual_results = loaded_vector_store.similarity_search(
    "Technology news",
    k=3
)

retriever_results = retriever.invoke("Technology news")

print("Manual Search:", len(manual_results))
print("Retriever:", len(retriever_results))

Manual Search: 3
Retriever: 3


# 🤖 Step 8: Build the Retrieval-Augmented Generation (RAG) Chain

In this step we combine:

- Retriever
- Prompt Template
- Groq LLM

The retriever fetches relevant context from the vector database, and the LLM generates answers grounded in that retrieved information.

In [45]:
from langchain_core.prompts import ChatPromptTemplate

from langchain_classic.chains.combine_documents import (
    create_stuff_documents_chain,
)

from langchain_classic.chains.retrieval import (
    create_retrieval_chain,
)

In [46]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

print("Groq LLM Initialized Successfully ✅")

Groq LLM Initialized Successfully ✅


In [47]:
prompt = ChatPromptTemplate.from_template("""
You are an intelligent AI News Assistant.

Answer the user's question ONLY using the provided context.

If the answer is not available in the context, reply:

"I couldn't find this information in the loaded news articles."

Context:
{context}

Question:
{input}

Answer:
""")

In [48]:
document_chain = create_stuff_documents_chain(
    llm=llm,
    prompt=prompt
)

print("Document Chain Created Successfully ✅")

Document Chain Created Successfully ✅


In [49]:
rag_chain = create_retrieval_chain(
    retriever,
    document_chain
)

print("RAG Chain Created Successfully ✅")

RAG Chain Created Successfully ✅


In [50]:
response = rag_chain.invoke(
    {
        "input": "What is today's top news?"
    }
)

In [51]:
print(response["answer"])

The top news is that Morocco has become the first team to reach the World Cup quarter-finals, with co-hosts Canada exiting the competition.


In [52]:
response.keys()

dict_keys(['input', 'context', 'answer'])

In [53]:
len(response["context"])

3

In [54]:
for i, doc in enumerate(response["context"], start=1):
    print("=" * 80)
    print(f"Document {i}")
    print("=" * 80)
    print(doc.page_content[:500])
    print()

Document 1
as Morocco become the first team to reach the World Cup quarter-finals, with co-hosts Canada exiting the competition.Most watched1BBC confronts head of Meta in India over child sex abuse ads2Why dangerous teen sex choking is raising alarm3Funeral of Iran's former supreme leader 'intensely political moment'4Watch: Taylor Swift and Travis Kelce wed in NYC - here's how the day unfolded5BBC in Tehran as mourners gather for former supreme leader's funeralAlso in newsNigeria says two nationals killed 

Document 2
"all the girls with ruffled socks and chubby cheeks".20 hrs agoTennisHow Iran's new regime is very different to what came beforeKhamenei's funeral is another reminder of the change Iran has seen, but what does its new leadership want?11 hrs agoBBC InDepthEvacuations in Guam as super typhoon Bavi approachesThe storm is forecast to bring winds in excess of 160mph and waves nearly 11m high when it makes landfall on Monday.7 hrs agoScience & EnvironmentLIVEWorld Cup 2026: Eng

In [55]:
questions = [
    "What is today's biggest news?",
    "What happened in sports?",
    "Any business news?",
    "What is happening in technology?"
]

for question in questions:

    print("\n" + "=" * 80)
    print("Question:", question)
    print("=" * 80)

    response = rag_chain.invoke({"input": question})

    print(response["answer"])


Question: What is today's biggest news?
I couldn't find this information in the loaded news articles.

Question: What happened in sports?
In sports, several events occurred: 

- England will face Mexico in the World Cup last 16 at The Azteca.
- The Premier League feud between Manchester City's Erling Haaland and Arsenal defender Gabriel is going global as Norway meets Brazil at the World Cup.
- Charles Leclerc won the British GP from George Russell and Lewis Hamilton.
- France survived Paraguay's 'disgraceful' and 'embarrassing' dark arts in the World Cup.
- Morocco reached the quarter-finals, and co-hosts Canada exited the World Cup.
- England is facing Australia in the Women's T20 World Cup final at Lord's.
- Wimbledon day seven is ongoing.

Question: Any business news?
I couldn't find this information in the loaded news articles.

Question: What is happening in technology?
I couldn't find this information in the loaded news articles.


In [ ]:
while True:

    question = input("\nAsk News > ")

    if question.lower() == "exit":
        break

    response = rag_chain.invoke(
        {
            "input": question
        }
    )

    print("\nAnswer:\n")
    print(response["answer"])


Answer:

It seems like you haven't asked a question yet. Please go ahead and ask your question, and I'll do my best to answer it based on the provided context.

Answer:

It seems like you haven't asked a question yet. Please go ahead and ask your question, and I'll do my best to answer it based on the provided context.
